# POD

`pod.pod` for one field and `pod.pod_multi` for several fields sharing time coefficients.

Data: GLORYS04 SSH (364 days) and surface U and V in the Gulf Stream (731 days, 30-45N, 75-45W). Animations show the anomaly beside its reconstruction.

In [ ]:
import numpy as np
from showcase import *
from modredkit import dmd, pod

RANK = 50

## SSH

Area-weighted, so modes are orthonormal over the ocean surface.

In [ ]:
A, area, wet = load_ssh()
print("SSH anomalies", A.shape)
modes, S, coeffs, _ = pod.pod(A, weights=area, remove_mean=False)
for e in (0.5, 0.9, 0.99):
    print(f"{e:.0%} of the variance needs {pod.rank_from_energy(S, e)} modes")
for r in (10, RANK, 100):
    print(f"r={r}: relative error {relative_error(modes[:, :r] @ coeffs[:r], A):.3f}")
show_variance(S, "SSH: POD variance captured")
for k in range(2):
    show_maps([modes[:, k]], [f"SSH mode {k+1}, {S[k]**2/np.sum(S**2):.1%} of variance"], [wet])

In [ ]:
rec = modes[:, :RANK] @ coeffs[:RANK]
animate([A, rec], ["SSH anomaly", "POD, 50 modes"], wet, units="m")

## Gulf Stream U and V

One joint decomposition, not weighted.

In [ ]:
U, V, wet_u, wet_v = load_gulfstream()
Z = np.vstack((U, V))
nu = U.shape[0]
print("U", U.shape, "V", V.shape)
ms, S, coeffs, _ = pod.pod_multi([U, V], remove_mean=False)
for e in (0.5, 0.9, 0.99):
    print(f"{e:.0%} of the variance needs {pod.rank_from_energy(S, e)} modes")
print(f"r={RANK}: relative error {relative_error(np.vstack(ms)[:, :RANK] @ coeffs[:RANK], Z):.3f}")
show_variance(S, "Gulf Stream: joint POD variance captured")
for k in range(2):
    show_maps([ms[0][:, k], ms[1][:, k]], [f"U, mode {k+1}, {S[k]**2/np.sum(S**2):.1%}", f"V, mode {k+1}"], [wet_u, wet_v])

In [ ]:
rec = np.vstack(ms)[:, :RANK] @ coeffs[:RANK]
animate([U, rec[:nu]], ["U anomaly", "POD, 50 modes"], wet_u, step=12, units="m/s")